In [7]:
from pathlib import Path
import geopandas as gpd

# -----------------------------
# paths
# -----------------------------
FP_VERDES_RAW = Path("../data/raw/areas_verdes_penalolen.gpkg")
FP_VERDES_OUT = Path("../data/processed/destinations/verdes_points.gpkg")

# -----------------------------
# cargar
# -----------------------------
verdes = gpd.read_file(FP_VERDES_RAW)

print("Antes:", verdes.shape)
print(verdes.geom_type.value_counts())

# -----------------------------
# separar multipartes
# -----------------------------
verdes = verdes.explode(index_parts=False).reset_index(drop=True)

print("Después de explode:", verdes.shape)
print(verdes.geom_type.value_counts())

# -----------------------------
# limpiar geometrías
# -----------------------------
verdes = verdes[verdes.geometry.notna()].copy()
verdes = verdes[~verdes.geometry.is_empty].copy()
verdes = verdes[verdes.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

# -----------------------------
# filtro opcional por área
# ajusta o comenta esta parte si no quieres filtrar
# -----------------------------
verdes["area_m2"] = verdes.geometry.area
verdes = verdes[verdes["area_m2"] >= 250].copy()

print("Después de filtro por área:", verdes.shape)



Antes: (1, 2)
MultiPolygon    1
Name: count, dtype: int64
Después de explode: (604, 2)
Polygon    604
Name: count, dtype: int64
Después de filtro por área: (525, 3)


In [8]:
# -----------------------------
# convertir a puntos representativos
# -----------------------------
verdes["geometry"] = verdes.representative_point()

# id único
verdes = verdes.reset_index(drop=True)
verdes["dest_id"] = [f"verde_{i}" for i in range(len(verdes))]

# dejar solo lo necesario
verdes_points = verdes[["dest_id", "area_m2", "geometry"]].copy()

print("Resultado final:", verdes_points.shape)
print(verdes_points.head())

# -----------------------------
# exportar
# -----------------------------
FP_VERDES_OUT.parent.mkdir(parents=True, exist_ok=True)
verdes_points.to_file(FP_VERDES_OUT, driver="GPKG")

print("Exportado en:", FP_VERDES_OUT)

Resultado final: (525, 3)
   dest_id      area_m2                        geometry
0  verde_0   826.331638  POINT (352357.430 6291031.407)
1  verde_1   441.391564  POINT (352803.387 6291060.319)
2  verde_2  6230.745185  POINT (352396.376 6291102.515)
3  verde_3   313.055953  POINT (353245.555 6291080.487)
4  verde_4   307.707879  POINT (353249.878 6291129.763)
Exportado en: ..\data\processed\destinations\verdes_points.gpkg


,id,geometry,green_id,dest_point
0,PA BUEN PL 1500,"POLYGON ((352369.969 6291004.059, 352345.516 6...",green_0,POINT (352357.430 6291031.407)
1,PA BUEN PL 1500,"POLYGON ((352783.811 6291039.403, 352746.964 6...",green_1,POINT (352725.307 6291043.461)
2,PA BUEN PL 1500,"POLYGON ((352676.850 6291046.318, 352644.305 6...",green_2,POINT (352645.064 6291049.202)
3,PA BUEN PL 1500,"POLYGON ((352796.900 6291055.057, 352796.900 6...",green_3,POINT (352803.387 6291060.319)
4,PA BUEN PL 1500,"POLYGON ((352392.492 6291010.494, 352369.969 6...",green_4,POINT (352396.376 6291102.515)
...,...,...,...,...
599,PA BUEN PL 1500,"POLYGON ((358779.723 6296496.945, 358839.714 6...",green_599,POINT (358806.522 6296483.494)
600,PA BUEN PL 1500,"POLYGON ((358746.635 6296496.636, 358768.590 6...",green_600,POINT (358753.328 6296487.050)
601,PA BUEN PL 1500,"POLYGON ((358925.995 6296598.174, 358922.935 6...",green_601,POINT (358941.636 6296613.855)
602,PA BUEN PL 1500,"POLYGON ((359134.261 6296689.474, 359134.276 6...",green_602,POINT (359142.138 6296687.249)
